In [9]:
import os 
import rosalia as rs
import pandas as pd
import numpy as np
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from astropy.io import fits
plt.style.use(os.path.dirname(rs.__file__) + "/style/presi_style.mplstyle")

In [10]:
# def get_thermal_image():
thermal_df = pd.read_csv("/Users/aborlaff/NASA/ROSALIA_DEPOT/THERMAL/F146_thermal.txt", delimiter="/s")


import numpy as np

def read_sca_tables(filename):
    sca_data = {}
    current_key = None
    current_rows = []

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Detect SCA headers (e.g., "SCA1", "SCA 8", "SCA 10")
            if line.startswith("SCA"):
                # Save previous block if present
                if current_key is not None and current_rows:
                    sca_data[current_key] = np.array(current_rows, dtype=float)
                    current_rows = []

                # Normalize SCA name (remove spaces)
                
                current_key = line.replace(" ", "")
                SCAi = str(int(current_key.replace("SCA",""))).zfill(2)
                current_key = "SCA"+SCAi
                continue

            # Otherwise it's a row of numbers
            numbers = line.split()
            current_rows.append([float(x) for x in numbers])

        # Store last table after loop ends
        # Flip the y-axis. origin=bottom.
        if current_key is not None and current_rows:
            # print(np.array(current_rows, dtype=float).shape)
            sca_data[current_key] = np.array(current_rows, dtype=float) # np.flip(np.array(current_rows, dtype=float), axis=0)
            # sca_data[current_key] = np.flip(np.array(current_rows, dtype=float), axis=0)

        for i in range(18):
            key = "SCA" + str(i+1).zfill(2)
            sca_data[key] = np.flip(sca_data[key], axis=0)
    return sca_data


def get_thermal_background(bandpass, SCA):
    # Example usage:
    filename = "/Users/aborlaff/NASA/ROSALIA_DEPOT/THERMAL/"+bandpass+"_thermal.txt"   # your text file
    sca_tables = read_sca_tables(filename)
    import numpy as np
    from scipy.ndimage import zoom

    target_size = 4088
    SCAi = str(SCA).zfill(2)
    SCAkey = "SCA" + SCAi
    zoom_factor = target_size / sca_tables[SCAkey].shape[0]

    upscaled = zoom(sca_tables[SCAkey], zoom_factor, order=3)  # cubic interpolation
    return(upscaled)   # (4088, 4088)



# Print shape of each table:
# for key, arr in sca_tables.items():
#     print(key, arr.shape)
F146_SCA1_thermal  = get_thermal_background(bandpass="F146", SCA=1)
plt.imshow(F146_SCA1_thermal, origin="lower")
plt.colorbar()





KeyboardInterrupt: 

In [ ]:
name = "/Users/aborlaff/NASA/ROSALIA/notebooks/MOCK/PR_mock_F184_Roman_RA_088.239_DEC_-48.533_MJD_61375.00000_PA_098.23_stray.fits"
stray_fits = fits.open(name)

for i in range(18):
    F146_SCAi_thermal  = get_thermal_background(bandpass="F146", SCA=i+1)
    stray_fits[i+1].data = F146_SCAi_thermal
stray_fits.verify("silentfix")
thermal_name = name.replace("_stray.fits", "_thermal.fits")
stray_fits.writeto(thermal_name, overwrite=True)



In [ ]:
# Make the scaled model and the summary plot.
thermal_name = name.replace("_stray.fits", "_thermal.fits")

drz_name, scaled_drz_name = rs.utils.run_swarp(pattern=thermal_name, 
                                               outname=thermal_name.replace(".fits","_drz.fits"), scale=0.11)


In [ ]:
def interpolate_subarrays(x, y, z, frame=(4088, 4088), method="linear"):
    """
    Interpolate scattered  onto the pixel grid of a FITS image
    using scipy.interpolate.RBFInterpolator (supports extrapolation).
    """

    from scipy.interpolate import RBFInterpolator
    from astropy.io import fits
    from astropy.wcs import WCS
    import numpy as np

    # Load FITS & WCS
    target_fits = fits.open(target_name)
    target_header = target_fits[target_ext].header
    w = WCS(header=target_header, fobj=target_fits, naxis=2)

    # Input point array for RBFInterpolator
    ori_points = np.column_stack((ra, dec))

    # Target pixel grid
    X = np.arange(target_header["NAXIS1"])
    Y = np.arange(target_header["NAXIS2"])
    XX, YY = np.meshgrid(X, Y)
    ra2, dec2 = w.wcs_pix2world(XX, YY, 0)
    if method=="nearest":
        from scipy.interpolate import griddata
        ZZ = griddata(ori_points, z, (ra2, dec2), method=method)
        if mask_original_nan: ZZ[np.isnan(target_fits[target_ext].data)] = np.nan
        return(ZZ)
    
    else:
        # Convert pixel grid → sky coordinates
        target_points = np.column_stack((ra2.ravel(), dec2.ravel()))

        # Build RBF model
        # Note: smoothing=0 gives pure interpolation. Adjust if needed to reduce noise.
        rbf = RBFInterpolator(ori_points, z, kernel=method, smoothing=0.0)

        # Evaluate on target grid
        ZZ = rbf(target_points).reshape(XX.shape)

        # Mask original NaNs
        if mask_original_nan:
            ZZ[np.isnan(target_fits[target_ext].data)] = np.nan
        return(ZZ)


In [ ]:
df = pd.read_excel('/Users/aborlaff/NASA/ROSALIA_DEPOT/THERMAL/Flight TSE 2026-02-26.xlsx', sheet_name=None, header=None)

In [ ]:
n = 64 # 
count = 1
thermal_df = np.zeros((18,3,64))
i = 0
min_row = 1
max_row = 65
while i < 18:
    label = "SCA" + str(i+1).zfill(2)
    #print(i*(n+count))
    x = df["W146 8x8"].iloc[min_row:max_row,0]
    y = df["W146 8x8"].iloc[min_row:max_row,1]
    t = df["W146 8x8"].iloc[min_row:max_row,2]
    
    thermal_df[i,0,:] = x
    thermal_df[i,1,:] = y
    thermal_df[i,2,:] = t

    i = i + 1

    print(t)
    min_row = min_row + 64 + 1
    max_row = min_row + 64
    print(thermal_df)

In [ ]:
plt.scatter(thermal_df[1,0,:], thermal_df[1,1,:], s=10000*(thermal_df[1,2,:]-np.min(thermal_df[1,2,:])))

In [ ]:
# Get the locations of the subarrays in the selected SCA.

for i in range(18):
    subarray_locations_db = rs.roman.get_subarray_locations(SCA=i+1, verbose=False)
    plt.imshow(subarray_locations_db["mask"])
    plt.show()

In [ ]:
subarray_locations_db["xmin"]

In [ ]:

SCA01 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA02 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA03 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")

SCA04 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA05 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA06 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")

SCA07 = np.array(df["W146 8x8"].iloc[1:9,     6:14], dtype="float32")
SCA08 = np.array(df["W146 8x8"].iloc[11:11+8, 6:14], dtype="float32")
SCA09 = np.array(df["W146 8x8"].iloc[20:20+8, 6:14], dtype="float32")

SCA10 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA11 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA12 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")

SCA13 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA14 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")
SCA15 = np.array(df["W146 8x8"].iloc[1:9, 6:14], dtype="float32")

SCA16 = np.array(df["W146 8x8"].iloc[1:9,     6:14], dtype="float32")     #  
SCA17 = np.array(df["W146 8x8"].iloc[11:11+8, 6:14], dtype="float32") # 
SCA18 = np.array(df["W146 8x8"].iloc[20:20+8, 6:14], dtype="float32") # 



In [ ]:
plt.imshow(df["W146 8x8"].iloc[1:9,     20:40],)